In [ ]:
import io, os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.signal import lfilter
import torch

warnings.filterwarnings("ignore", message=".*NumPy.*")

PROJECT_ROOT = Path("/Users/luorix/Desktop/MetaMobility Lab (CMU)/projects/MeMo-IK-ID")
sys.path.insert(0, str(PROJECT_ROOT))
from dataset import IK_DOF_NAMES, MOMENT_NAMES, _compute_velocity
from model import TCN

## 1 · Configuration

In [ ]:
OPENSIM_ROOT   = Path("/Users/luorix/Desktop/MetaMobility Lab (CMU)/projects/opensim-processing/data/AB01_Jinwoo")
CHECKPOINT     = PROJECT_ROOT / "runs/0507_ik_id_all/best_model.pt"

SUBJECT_MASS_KG    = 88.0      # model output (N·m/kg) × mass → N·m for comparison
ID_LPF_CUTOFF_HZ   = 6.0       # must match out_lpf_hz in controller YAML
ID_LPF_ORDER       = 4         # cascaded 1st-order EMA stages (causal)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── model output channel order (after unilateral-paired bilateral inference) ──
#   Right side first, then Left; each side: hip_flexion, knee_angle, ankle_angle
OUTPUT_LABELS = [
    ("hip_flexion_r",  "Hip Flexion R"),
    ("knee_angle_r",   "Knee Angle R"),
    ("ankle_angle_r",  "Ankle Angle R"),
    ("hip_flexion_l",  "Hip Flexion L"),
    ("knee_angle_l",   "Knee Angle L"),
    ("ankle_angle_l",  "Ankle Angle L"),
]
N_OUT = len(OUTPUT_LABELS)

# Corresponding OpenSim ID .sto column names (N·m, not normalised by mass)
ID_COLS = [
    "hip_flexion_r_moment",
    "knee_angle_r_moment",
    "ankle_angle_r_moment",
    "hip_flexion_l_moment",
    "knee_angle_l_moment",
    "ankle_angle_l_moment",
]

# ── colour palette ─────────────────────────────────────────────────────────────
PALETTE = {
    "id":       "#2196F3",
    "model":    "#F44336",
    "residual": "#9C27B0",
}
# Per-channel colours (alternating warm/cool for R/L)
CHAN_COLORS_MODEL = ["#E53935","#D81B60","#8E24AA","#1E88E5","#039BE5","#00ACC1"]
CHAN_COLORS_ID    = ["#90CAF9","#CE93D8","#A5D6A7","#FFCC80","#B0BEC5","#80CBC4"]

# ── trial mapping: stem → (exo_subfolder, opensim_condition_stem) ─────────────
TRIALS = {
    "awinda_LG_0p8mps": ("awinda", "LG_0p8mps"),
    "awinda_LG_1p2mps": ("awinda", "LG_1p2mps"),
    "awinda_LG_1p6mps": ("awinda", "LG_1p6mps"),
    "awinda_RA_0p8mps": ("awinda", "RA_0p8mps"),
    "awinda_RD_0p8mps": ("awinda", "RD_0p8mps"),
}

print("Trial manifest:")
for stem, (exo, cond) in TRIALS.items():
    ik_ok  = (OPENSIM_ROOT / exo / "ik" / f"{cond}_ik.mot").exists()
    id_ok  = (OPENSIM_ROOT / exo / "id" / f"{cond}_id.sto").exists()
    print(f"  {'✓' if ik_ok else '✗'} ik  {'✓' if id_ok else '✗'} id   {stem}")
print(f"\nCheckpoint: {'✓' if CHECKPOINT.exists() else '✗'}  {CHECKPOINT}")

## 2 · Helper functions

In [ ]:
def parse_opensim_file(path: Path) -> pd.DataFrame:
    """Read an OpenSim .sto or .mot file; returns a DataFrame indexed by time."""
    with open(path) as fh:
        for i, line in enumerate(fh):
            if line.strip() == "endheader":
                header_end = i
                break
    df = pd.read_csv(path, sep=r"\s+", skiprows=header_end + 1)
    return df.set_index("time")


def lpf_id(signal: np.ndarray, fs: float,
           cutoff_hz: float = ID_LPF_CUTOFF_HZ,
           order: int = ID_LPF_ORDER) -> np.ndarray:
    """
    Causal cascaded EMA low-pass — replicates _CausalLowPass from cascade.py.
    alpha = dt / (tau + dt),  tau = 1/(2π·cutoff_hz).
    Applied `order` times in series; initialised to first sample.
    """
    dt    = 1.0 / float(fs)
    tau   = 1.0 / (2.0 * np.pi * float(cutoff_hz))
    alpha = dt / (tau + dt)
    b = np.array([alpha])
    a = np.array([1.0, -(1.0 - alpha)])
    y = signal.astype(float).copy()
    for _ in range(order):
        zi = np.array([y[0] * (1.0 - alpha)])
        y, _ = lfilter(b, a, y, zi=zi)
    return y.astype(signal.dtype)


def lpf_id_multichannel(signals: np.ndarray, fs: float) -> np.ndarray:
    """Apply lpf_id independently to each column of an (T, C) array."""
    return np.stack([lpf_id(signals[:, c], fs) for c in range(signals.shape[1])], axis=1)


print("Helpers defined.")

## 3 · Load model checkpoint

In [ ]:
ckpt = torch.load(str(CHECKPOINT), map_location=DEVICE, weights_only=False)

cfg = ckpt["model_config"]
model = TCN(
    n_input_channels=cfg["n_input_channels"],
    n_output_channels=cfg["n_output_channels"],
    hidden_channels=cfg["hidden_channels"],
    n_blocks=cfg["n_blocks"],
    kernel_size=cfg["kernel_size"],
    dropout=cfg["dropout"],
).to(DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

WINDOW_SIZE     = int(ckpt["window_size"])
INPUT_INDICES   = list(ckpt["input_indices"])    # [6,9,10,13,16,17]
MOMENT_INDICES  = list(ckpt["moment_indices"])   # [3,12,14,6,13,15]
DOF_NAMES       = list(ckpt["dof_names"])        # ['hip_flexion','knee_angle','ankle_angle']

norm     = ckpt["normalization"]
POS_MEAN = np.array(norm["pos_mean"])   # shape (23,)
POS_STD  = np.array(norm["pos_std"])
VEL_MEAN = np.array(norm["vel_mean"])
VEL_STD  = np.array(norm["vel_std"])

# Split into R / L halves for unilateral-paired inference
H = len(INPUT_INDICES) // 2
INPUT_IDX_R  = INPUT_INDICES[:H]   # [6,9,10]  → hip_r, knee_r, ankle_r
INPUT_IDX_L  = INPUT_INDICES[H:]   # [13,16,17] → hip_l, knee_l, ankle_l

print(f"Model: {cfg['n_input_channels']}→{cfg['n_output_channels']} channels, "
      f"window={WINDOW_SIZE}, blocks={cfg['n_blocks']}, hidden={cfg['hidden_channels']}")
print(f"Input DoFs R : {[IK_DOF_NAMES[i] for i in INPUT_IDX_R]}")
print(f"Input DoFs L : {[IK_DOF_NAMES[i] for i in INPUT_IDX_L]}")
print(f"Output moments: {[MOMENT_NAMES[i] for i in MOMENT_INDICES]}")
print(f"Device: {DEVICE}")

## 4 · Inference functions

In [ ]:
@torch.no_grad()
def causal_infer_unilateral(pos_in: np.ndarray, vel_in: np.ndarray,
                             pos_mean_sel: np.ndarray, pos_std_sel: np.ndarray,
                             vel_mean_sel: np.ndarray, vel_std_sel: np.ndarray) -> np.ndarray:
    """
    Run causal TCN inference for one side (R or L).
    Inputs are raw (un-normalised) position (rad) and velocity (rad/s) arrays,
    each shape (T, 3).  Returns predictions shape (T, 3) in model units (N·m/kg).
    """
    pos_n = (pos_in - pos_mean_sel) / pos_std_sel   # normalise
    vel_n = (vel_in - vel_mean_sel) / vel_std_sel
    x     = np.concatenate([pos_n, vel_n], axis=1).astype(np.float32)  # (T, 6)

    n   = x.shape[0]
    W   = WINDOW_SIZE
    n_c = model.network[-1].weight.shape[0] if hasattr(model, 'network') else cfg["n_output_channels"]
    pred = np.zeros((n, cfg["n_output_channels"]), dtype=np.float64)

    def _fwd(start):
        xw = torch.from_numpy(x[start:start+W].T).unsqueeze(0).to(DEVICE)  # (1, 6, W)
        return model(xw).squeeze(0).detach().cpu().numpy().T               # (W, n_out)

    pw0 = _fwd(0)
    for g in range(W - 1):
        pred[g] = pw0[g]
    for start in range(n - W + 1):
        pred[start + W - 1] = _fwd(start)[W - 1]

    return pred.astype(np.float32)   # (T, 3)


def run_inference(pos_all: np.ndarray, vel_all: np.ndarray) -> np.ndarray:
    """
    Bilateral inference via unilateral-paired strategy.
    pos_all, vel_all : (T, 23) — full IK position/velocity arrays (radians).
    Returns pred : (T, 6)  in N·m/kg, order: [hip_r, knee_r, ankle_r, hip_l, knee_l, ankle_l].
    """
    pred_r = causal_infer_unilateral(
        pos_all[:, INPUT_IDX_R], vel_all[:, INPUT_IDX_R],
        POS_MEAN[INPUT_IDX_R], POS_STD[INPUT_IDX_R],
        VEL_MEAN[INPUT_IDX_R], VEL_STD[INPUT_IDX_R],
    )
    pred_l = causal_infer_unilateral(
        pos_all[:, INPUT_IDX_L], vel_all[:, INPUT_IDX_L],
        POS_MEAN[INPUT_IDX_L], POS_STD[INPUT_IDX_L],
        VEL_MEAN[INPUT_IDX_L], VEL_STD[INPUT_IDX_L],
    )
    return np.concatenate([pred_r, pred_l], axis=1)   # (T, 6)


print("Inference functions defined.")

## 5 · Pre-load all trials

Reads IK, runs inference, reads ID, and applies the 6 Hz causal LPF to ID moments. IK and ID share the same OpenSim time axis — no GPIO sync needed.

In [ ]:
TRIAL_DATA = {}

for stem, (exo, cond) in TRIALS.items():
    ik_path = OPENSIM_ROOT / exo / "ik" / f"{cond}_ik.mot"
    id_path = OPENSIM_ROOT / exo / "id" / f"{cond}_id.sto"

    if not ik_path.exists() or not id_path.exists():
        print(f"[SKIP] {stem} — missing file(s)")
        continue

    print(f"Processing {stem} …", end=" ", flush=True)

    # ── IK: load, degrees → radians, compute velocity ─────────────────────────
    ik_df   = parse_opensim_file(ik_path)
    t_ik    = ik_df.index.to_numpy(float)
    # Columns may not include all IK_DOF_NAMES; fill missing with 0
    pos_deg = np.zeros((len(t_ik), len(IK_DOF_NAMES)), dtype=np.float64)
    for j, name in enumerate(IK_DOF_NAMES):
        if name in ik_df.columns:
            pos_deg[:, j] = ik_df[name].to_numpy(float)
    pos_rad = np.deg2rad(pos_deg)          # (T, 23) radians
    vel_rad = _compute_velocity(pos_rad, t_ik)   # (T, 23) rad/s (B-spline for 3D joints)

    # ── Run bilateral causal inference ────────────────────────────────────────
    pred_nmpkg = run_inference(pos_rad, vel_rad)   # (T, 6)  N·m/kg
    pred_nm    = pred_nmpkg * SUBJECT_MASS_KG      # (T, 6)  N·m

    # ── ID: load, apply 6 Hz causal LPF ──────────────────────────────────────
    id_df      = parse_opensim_file(id_path)
    t_id       = id_df.index.to_numpy(float)
    id_fs      = 1.0 / float(np.median(np.diff(t_id)))

    id_raw = np.zeros((len(t_id), N_OUT), dtype=np.float64)
    for c, col in enumerate(ID_COLS):
        if col in id_df.columns:
            id_raw[:, c] = id_df[col].to_numpy(float)
        else:
            id_raw[:, c] = np.nan
    id_filt = lpf_id_multichannel(id_raw, fs=id_fs)   # (T_id, 6)  N·m

    # ── Align IK and ID on their shared time axis ─────────────────────────────
    t_lo = max(t_ik[0],  t_id[0])
    t_hi = min(t_ik[-1], t_id[-1])
    mask = (t_ik >= t_lo) & (t_ik <= t_hi)
    t_common   = t_ik[mask]
    pred_common = pred_nm[mask]

    id_interp = np.stack([
        interp1d(t_id, id_filt[:, c], kind="linear",
                 bounds_error=False, fill_value=np.nan)(t_common)
        for c in range(N_OUT)
    ], axis=1)   # (T_common, 6)

    # ── Per-channel metrics ───────────────────────────────────────────────────
    metrics = []
    for c in range(N_OUT):
        v = ~np.isnan(id_interp[:, c])
        if v.sum() > 1:
            rmse = float(np.sqrt(np.mean((pred_common[v, c] - id_interp[v, c])**2)))
            r2   = float(np.corrcoef(pred_common[v, c], id_interp[v, c])[0, 1])**2
        else:
            rmse, r2 = np.nan, np.nan
        metrics.append({"rmse": rmse, "r2": r2})

    TRIAL_DATA[stem] = {
        "stem":      stem,
        "exo":       exo,
        "cond":      cond,
        "t":         t_common,
        "t_rel":     t_common - t_common[0],
        "pred_nm":   pred_common,    # (T, 6)
        "id_nm":     id_interp,      # (T, 6)
        "metrics":   metrics,        # list of {rmse, r2} per channel
    }

    mean_r2   = float(np.nanmean([m["r2"]   for m in metrics]))
    mean_rmse = float(np.nanmean([m["rmse"] for m in metrics]))
    dur = float(t_common[-1] - t_common[0])
    print(f"ok  ({dur:.1f} s, mean R²={mean_r2:.3f}, mean RMSE={mean_rmse:.2f} N·m)")

print(f"\nLoaded {len(TRIAL_DATA)} / {len(TRIALS)} trials.")

## 6 · Interactive comparison

In [ ]:
_first_stem = next(iter(TRIAL_DATA))
_first_d    = TRIAL_DATA[_first_stem]

trial_dropdown = widgets.Dropdown(
    options=list(TRIAL_DATA.keys()),
    value=_first_stem,
    description="Trial:",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "80px"},
)

time_slider = widgets.FloatRangeSlider(
    value=[_first_d["t_rel"][0], _first_d["t_rel"][-1]],
    min=_first_d["t_rel"][0],
    max=_first_d["t_rel"][-1],
    step=0.5,
    description="Time (s):",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "80px"},
    continuous_update=False,
    readout_format=".1f",
)

show_residual = widgets.ToggleButton(
    value=False,
    description="Show residuals",
    button_style="",
    icon="signal",
    layout=widgets.Layout(width="160px"),
)

out = widgets.Output()


# ── draw ──────────────────────────────────────────────────────────────────────
def _draw(stem: str, t0: float, t1: float, residuals: bool) -> None:
    d    = TRIAL_DATA[stem]
    mask = (d["t_rel"] >= t0) & (d["t_rel"] <= t1)
    t    = d["t_rel"][mask]
    pred = d["pred_nm"][mask]   # (T_win, 6)
    id_m = d["id_nm"][mask]     # (T_win, 6)

    lpf_tag = f"{ID_LPF_CUTOFF_HZ:.0f} Hz LPF"
    n_rows   = N_OUT * 2 if residuals else N_OUT
    row_h    = 2.2
    fig_h    = n_rows * row_h + 0.6

    # 3×2 layout: rows = joints (hip/knee/ankle), cols = sides (R/L)
    n_joints = 3
    n_sides  = 2
    n_main   = n_joints * n_sides if not residuals else n_joints * n_sides * 2

    fig, axs = plt.subplots(
        n_joints * (2 if residuals else 1), n_sides,
        figsize=(16, fig_h),
        gridspec_kw={"hspace": 0.55, "wspace": 0.25},
        sharex=True,
    )
    if axs.ndim == 1:
        axs = axs.reshape(-1, n_sides)

    joint_names = ["Hip Flexion", "Knee Angle", "Ankle Angle"]
    side_names  = ["Right", "Left"]

    for j in range(n_joints):
        for s in range(n_sides):
            c_idx = j + s * n_joints   # channel: 0-2 = R, 3-5 = L
            label_key, label_str = OUTPUT_LABELS[c_idx]
            m   = metrics = d["metrics"][c_idx]

            # ── moment overlay row ───────────────────────────────────────────
            ax = axs[j * (2 if residuals else 1), s]
            ax.plot(t, id_m[:, c_idx],   color=PALETTE["id"],    lw=2.0,
                    label=f"OpenSim ID ({lpf_tag})")
            ax.plot(t, pred[:, c_idx],   color=PALETTE["model"], lw=1.8, ls="--",
                    label=f"Model (GT IK → TCN × {SUBJECT_MASS_KG:.0f} kg)")
            ax.set_ylabel("N·m", fontsize=10)
            ax.set_title(
                f"{joint_names[j]} — {side_names[s]}
"
                f"RMSE={m['rmse']:.2f} N·m   R²={m['r2']:.3f}",
                fontsize=9,
            )
            ax.axhline(0, color="gray", lw=0.6, ls=":")
            ax.spines[["top","right"]].set_visible(False)
            ax.tick_params(labelsize=9)
            if j == 0 and s == 0:
                ax.legend(fontsize=8, loc="upper right")

            # ── residual row (optional) ──────────────────────────────────────
            if residuals:
                ax_r = axs[j * 2 + 1, s]
                resid = pred[:, c_idx] - id_m[:, c_idx]
                ax_r.plot(t, resid, color=PALETTE["residual"], lw=1.2)
                ax_r.axhline(0, color="black", lw=0.7, ls=":")
                ax_r.set_ylabel("N·m", fontsize=9)
                ax_r.set_title("Residual (Model − ID)", fontsize=8)
                ax_r.spines[["top","right"]].set_visible(False)
                ax_r.tick_params(labelsize=9)

    # x-axis label on bottom row
    for s in range(n_sides):
        axs[-1, s].set_xlabel("Time (s)", fontsize=10)

    mean_r2   = float(np.nanmean([d["metrics"][c]["r2"]   for c in range(N_OUT)]))
    mean_rmse = float(np.nanmean([d["metrics"][c]["rmse"] for c in range(N_OUT)]))
    fig.suptitle(
        f"{stem}  |  GT IK → TCN → N·m vs OpenSim ID ({lpf_tag})
"
        f"Mean R² = {mean_r2:.3f}   Mean RMSE = {mean_rmse:.2f} N·m",
        fontsize=11, fontweight="bold",
    )

    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    buf.seek(0)
    out.clear_output(wait=True)
    with out:
        display(Image(data=buf.read()))


# ── callbacks ─────────────────────────────────────────────────────────────────
def _on_trial(change):
    d = TRIAL_DATA[change["new"]]
    time_slider.min   = float(d["t_rel"][0])
    time_slider.max   = float(d["t_rel"][-1])
    time_slider.value = [float(d["t_rel"][0]), float(d["t_rel"][-1])]
    _redraw()

def _redraw(*_):
    _draw(trial_dropdown.value, *time_slider.value, show_residual.value)

trial_dropdown.observe(_on_trial,  names="value")
time_slider.observe(_redraw,        names="value")
show_residual.observe(_redraw,      names="value")

controls = widgets.HBox(
    [trial_dropdown, show_residual],
    layout=widgets.Layout(align_items="center", gap="12px"),
)
display(controls, time_slider, out)
_redraw()

## 7 · Window statistics

Re-run after moving the slider.

In [ ]:
t0_s, t1_s = time_slider.value
stem_s     = trial_dropdown.value
d_s        = TRIAL_DATA[stem_s]
lpf_tag    = f"{ID_LPF_CUTOFF_HZ:.0f} Hz LPF"

mask_s   = (d_s["t_rel"] >= t0_s) & (d_s["t_rel"] <= t1_s)
pred_s   = d_s["pred_nm"][mask_s]   # (T_win, 6)
id_s_win = d_s["id_nm"][mask_s]     # (T_win, 6)

print(f"Trial  : {stem_s}")
print(f"Window : {t0_s:.1f} – {t1_s:.1f} s  ({t1_s-t0_s:.1f} s)")
print()
print(f"{'Channel':<30} {'RMSE [N·m]':>10} {'R²':>7} {'peak model':>11} {'peak ID':>9}")
print("-" * 72)
for c, (key, label) in enumerate(OUTPUT_LABELS):
    v = ~np.isnan(id_s_win[:, c])
    if v.sum() > 1:
        rmse_c = float(np.sqrt(np.mean((pred_s[v, c] - id_s_win[v, c])**2)))
        r2_c   = float(np.corrcoef(pred_s[v, c], id_s_win[v, c])[0, 1])**2
        pk_m   = float(np.nanmax(np.abs(pred_s[v, c])))
        pk_i   = float(np.nanmax(np.abs(id_s_win[v, c])))
    else:
        rmse_c = r2_c = pk_m = pk_i = np.nan
    print(f"  {label:<28} {rmse_c:>10.3f} {r2_c:>7.3f} {pk_m:>11.2f} {pk_i:>9.2f}")

mean_r2   = float(np.nanmean([float(np.corrcoef(pred_s[~np.isnan(id_s_win[:,c]),c],
                                                 id_s_win[~np.isnan(id_s_win[:,c]),c])[0,1])**2
                               for c in range(N_OUT) if (~np.isnan(id_s_win[:,c])).sum()>1]))
mean_rmse = float(np.nanmean([float(np.sqrt(np.mean((pred_s[~np.isnan(id_s_win[:,c]),c]
                                                      - id_s_win[~np.isnan(id_s_win[:,c]),c])**2)))
                               for c in range(N_OUT) if (~np.isnan(id_s_win[:,c])).sum()>1]))
print()
print(f"  {'Mean (all channels)':<28} {mean_rmse:>10.3f} {mean_r2:>7.3f}")

## 8 · All-trials summary (full overlap)

In [ ]:
rows = []
for stem, d in TRIAL_DATA.items():
    row = {"trial": stem, "exo": d["exo"], "condition": d["cond"],
           f"ID LPF": f"{ID_LPF_CUTOFF_HZ:.0f} Hz {ID_LPF_ORDER}-ord causal",
           "overlap [s]": round(float(d["t_rel"][-1] - d["t_rel"][0]), 1)}
    for c, (key, label) in enumerate(OUTPUT_LABELS):
        m = d["metrics"][c]
        row[f"RMSE_{key} [N·m]"] = round(m["rmse"], 2) if not np.isnan(m["rmse"]) else np.nan
        row[f"R²_{key}"]          = round(m["r2"],   3) if not np.isnan(m["r2"])   else np.nan
    row["mean_RMSE [N·m]"] = round(float(np.nanmean([d["metrics"][c]["rmse"] for c in range(N_OUT)])), 2)
    row["mean_R²"]          = round(float(np.nanmean([d["metrics"][c]["r2"]   for c in range(N_OUT)])), 3)
    rows.append(row)

pd.DataFrame(rows)